<div style="padding: 20px; background: linear-gradient(90deg, #11998e 0%, #38ef7d 100%); border-radius: 10px; color: white;">
    <h1 style="color: white; border-bottom: none;">🧠 Module 7.4: Self-RAG</h1>
    <p style="font-size: 1.2em; opacity: 0.9;">Giving the LLM complete agency over the RAG pipeline using Reflection Tokens.</p>
</div>

---

## 1. The Autonomous LLM

Self-RAG shifts control from hardcoded Python scripts to the LLM itself. We ask the LLM questions (called **Reflection Tokens**) at every stage of the pipeline:
1. **[Retrieve?]** - *"LLM, do you actually need to search the database to answer this, or do you already know?"*
2. **[Relevant?]** - *"LLM, are these retrieved documents actually useful?"*
3. **[Grounded?]** - *"LLM, is the answer you just wrote fully supported by the documents, or did you hallucinate?"*

Let's implement this pipeline.

### Course alignment and free-first stack

- Covers: Self-RAG style reflection gates for retrieve, relevance, and groundedness decisions.
- Runtime stack: Groq chat models for generation/evaluation when an LLM is needed, plus local Hugging Face sentence-transformers embeddings for retrieval.
- No paid OpenAI API key is required. Set `GROQ_API_KEY` only for notebooks that call an LLM; pure retrieval and embedding notebooks run locally after model weights are available.
- Current LangChain pattern: provider split packages such as `langchain_groq`, `langchain_huggingface`, and `langchain_chroma`, with runnable `.invoke()` APIs.


In [ ]:
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document
from dotenv import load_dotenv
import os

os.environ['TOKENIZERS_PARALLELISM'] = 'false'
load_dotenv()

docs = [
    Document(page_content="The speed of light in vacuum is approximately 299,792,458 metres per second."),
    Document(page_content="Einstein's theory of relativity shows mass and energy are equivalent: E=mc²."),
    Document(page_content="Photons are massless particles that always travel at the speed of light."),
]
embeddings = HuggingFaceEmbeddings(model_name=os.getenv("EMBEDDING_MODEL", "sentence-transformers/all-MiniLM-L6-v2"))
vs = Chroma.from_documents(docs, embeddings, collection_name="selfrag_demo")
print("Physics DB Loaded.")

## 2. Defining the Reflection Tokens
We use strict prompts to force `ChatGroq` to output boolean reflection tokens.

In [ ]:
groq_api_key = os.environ.get("GROQ_API_KEY")

if groq_api_key:
    llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0)
    
    def needs_retrieval(query: str) -> bool:
        prompt = ChatPromptTemplate.from_template("""
        Does answering this question strictly require looking up external knowledge?
        Answer exactly 'yes' or 'no'.
        Question: {query}
        """)
        result = (prompt | llm | StrOutputParser()).invoke({"query": query})
        return "yes" in result.lower()
        
    def is_relevant(query: str, doc_content: str) -> bool:
        prompt = ChatPromptTemplate.from_template("""
        Is this document relevant to the question? Answer exactly 'yes' or 'no'.
        Question: {query}
        Document: {doc}
        """)
        result = (prompt | llm | StrOutputParser()).invoke({"query": query, "doc": doc_content})
        return "yes" in result.lower()
        
    def is_grounded(answer: str, context: str) -> bool:
        prompt = ChatPromptTemplate.from_template("""
        Is this answer strictly supported by the context? Answer exactly 'yes' or 'no'.
        Context: {context}
        Answer: {answer}
        """)
        result = (prompt | llm | StrOutputParser()).invoke({"answer": answer, "context": context})
        return "yes" in result.lower()
else:
    print("GROQ_API_KEY missing.")

## 3. The Self-RAG Pipeline

In [ ]:
if groq_api_key:
    def self_rag(query: str):
        print(f"\n─── QUERY: {query} ───")
    
        # Reflection 1: Should we retrieve?
        retrieve = needs_retrieval(query)
        print(f"[Reflection 1: Retrieve?] → {retrieve}")
    
        context = ""
        if retrieve:
            candidates = vs.similarity_search(query, k=3)
            
            # Reflection 2: Are documents relevant?
            relevant = [d for d in candidates if is_relevant(query, d.page_content)]
            print(f"[Reflection 2: Relevant Docs?] → Kept {len(relevant)} out of {len(candidates)}")
            context = "\n".join(d.page_content for d in relevant)
    
        # Generate Answer
        gen_prompt = ChatPromptTemplate.from_template("""
        Answer the question{ctx_note}.
        {context}
        Question: {query}
        """)
        answer = (gen_prompt | llm | StrOutputParser()).invoke({
            "query": query, 
            "context": f"Context:\n{context}" if context else "",
            "ctx_note": " using the context below" if context else ""
        })
    
        # Reflection 3: Is answer grounded?
        grounded = is_grounded(answer, context) if context else True
        print(f"[Reflection 3: Grounded?] → {grounded}")
        print(f"\nFINAL ANSWER: {answer.strip()}")
    
    # --- Tests ---
    self_rag("What is the speed of light?") # Will trigger retrieval
    self_rag("What is your favourite colour?") # Will bypass retrieval!